In [33]:
for name in dir():
    if not name.startswith("_") and name not in ["In", "Out", "get_ipython", "exit", "quit"]:
        del globals()[name]


### sport和tech的tf_top1000丟給GAP
 跑混淆矩陣, 看分類的正確率

In [ ]:
import pandas as pd
data = pd.read_csv(r'C:\Users\No\Documents\GitHub\psychic-spoon\0 Meeting\sport_tech_bbcnews.csv').drop(columns='Unnamed: 0')
# 你把 list 存成 .csv 之後，pandas.read_csv 會把它當成 字串，不會還原回原本的 list。
# 所以
import ast
data['Tokens'] = data['Tokens'].apply(ast.literal_eval)
import numpy as np
np.random.seed(42)
data['filename'] = data.index.map(lambda i: f'{i+1:03}.txt')
data['filename'] = data['filename'].astype(str)

gap_df = pd.read_csv(r'C:\Users\No\Documents\GitHub\psychic-spoon\gap_doc_clusters.csv')
gap_df['filename'] = gap_df['filename'].astype(int).apply(lambda x: f"{x:03}.txt")

merged = pd.merge(data, gap_df, on='filename')
merged.head(3)

### sport 分對506筆doc, tech分對392筆doc, 正確率達0.98

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_true = le.fit_transform(merged['category'])+1
y_cluster = merged['gap_cluster'].astype(int).values

print(confusion_matrix(y_true, y_cluster))
print(classification_report(y_true, y_cluster, target_names=le.classes_))

---
---

### 畫圖

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

token_counts = Counter([t for tokens in data['Tokens'] for t in tokens])
valid_tokens = {t for t, c in token_counts.items() if c>=2}
merged['filteredTokens'] = data['Tokens'].apply(lambda ts: [t for t in ts if t in valid_tokens])
merged['filtered_content'] = merged['filteredTokens'].apply(lambda ts: ' '.join(ts))

In [ ]:
print(f'原本token數: {len(token_counts)}, 扣掉只出現1次的token數: {len(valid_tokens)}')

In [ ]:
vectorizer = TfidfVectorizer(max_features=1000)
X = vectorizer.fit_transform(merged['filtered_content']).toarray()
tsne = TSNE(n_components=2, random_state=42)
X_embedded = tsne.fit_transform(X)

plt.figure(figsize=(8, 6))
sns.scatterplot(data=merged, x=X_embedded[:, 0], y=X_embedded[:, 1], hue='gap_cluster', palette='tab10')
plt.title('t_SNE')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.tight_layout()
plt.show()

---
---

### 用LogisticRegression來學習, 哪些token是重要的

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(merged['filtered_content'])
y = merged['gap_cluster']


clf = LogisticRegression(max_iter=1000)
model_clf = clf.fit(X, y)

feature_names = vectorizer.get_feature_names_out()
coef = model_clf.coef_[0]

top_pos = np.argsort(coef)[-30:]
top_neg = np.argsort(coef)[:30]

print("Cluster 1 特有詞：")
print(feature_names[top_pos])

print("\nCluster 2 特有詞：")
print(feature_names[top_neg])


---
---

In [ ]:
from gensim.corpora import Dictionary
from gensim.models import LdaModel

def train_lda_for_cluster(cluster_id, num_topics=3, topn=30):
    tokens = merged[merged['gap_cluster'] == cluster_id]['filteredTokens'].tolist()
    dictionary = Dictionary(tokens)
    corpus = [dictionary.doc2bow(doc) for doc in tokens]
    lda = LdaModel(corpus=corpus, id2word=dictionary, num_topics=num_topics, passes=10, random_state=42, iterations=100)
    print(f"\n🟦 GAP Cluster {cluster_id} - Top {num_topics} Topics:")
    for i, topic in lda.print_topics(num_topics=num_topics, num_words=topn):
        print(f"Topic {i}: {topic}")
    return lda

lda_cluster_1 = train_lda_for_cluster(1)
lda_cluster_2 = train_lda_for_cluster(2)

🟨 新文件中的詞彙必須在原本的詞表中出現過
否則那部分詞會被忽略，向量稀疏，模型效果變差

若新文章出現很多從未見過的詞 → 模型會表現不佳

所以建議：詞表越大越穩定，但也越容易 overfit

🟨 模型只能預測原本訓練時學到的分類（如 cluster 1&2）
如果新來的文章語意和訓練資料完全不同，則分類可能錯誤，但這是分類模型的自然限制。

---
---

## Banknote資料

In [34]:
import pandas as pd

banknote = pd.read_csv(r"C:\Users\No\Documents\GitHub\psychic-spoon\DataSet\banknote.csv", sep='\t')
banknote['filename'] = [f'{i}' for i in range(1, 1372)]
banknote['filename'] = banknote['filename'].astype(int)
# banknote = banknote.drop(columns=['class']) 
# banknote.to_csv('banknote.csv', sep='\t', encoding='utf-8', index=False) 
gap_banknote_order = pd.read_csv(r"C:\Users\No\Documents\GitHub\psychic-spoon\DataSet\gap_banknote_order.txt", sep='\t')
raw='1  0  1  0  1  1  0  1  1  1  0  1  1  1  0  1  1  0  1  1  0  1  1  1  0  0  1  1  0  0  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  0  1  0  1  1  0  1  1  0  1  1  1  0  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  0  1  0  1  1  1  1  0  1  0  1  0  1  1  0  0  1  0  1  1  1  1  1  1  0  1  1  1  1  1  1  1  0  0  1  1  1  0  1  1  1  0  1  1  0  1  1  1  1  1  1  1  1  1  0  0  1  0  1  0  1  0  1  1  1  1  1  1  1  0  0  0  0  1  1  1  1  1  1  0  1  1  1  1  0  0  0  1  1  0  1  1  0  1  1  0  1  1  1  1  0  1  0  0  0  1  0  1  1  1  1  1  0  1  1  0  1  1  1  1  1  1  1  0  1  1  1  0  1  1  1  0  1  1  1  1  0  1  1  1  0  1  0  1  0  1  0  0  1  1  1  0  1  1  1  0  1  1  1  0  0  0  1  1  0  1  1  1  1  0  0  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  0  1  1  0  0  0  0  1  0  0  1  1  1  1  1  1  0  0  1  1  1  0  1  0  1  1  1  1  1  1  0  1  1  0  0  1  1  1  0  1  1  0  1  0  1  0  1  1  1  0  1  1  1  1  0  1  1  1  0  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  0  0  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  0  1  1  0  1  1  1  1  1  0  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  0  0  1  0  1  1  1  1  1  1  0  1  1  0  1  0  1  0  0  0  1  0  1  1  1  0  1  1  1  0  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  0  1  1  1  0  0  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  0  1  0  1  1  1  1  1  1  1  1  1  1  0  1  1  0  1  0  0  0  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  0  0  1  1  1  1  1  1  1  1  1  1  1  0  0  0  0  1  0  1  1  1  1  0  1  0  1  0  1  1  1  0  0  1  1  1  0  0  1  0  1  1  0  1  1  1  1  0  1  0  1  1  1  1  1  0  0  1  1  1  1  0  1  0  1  1  1  1  0  0  1  0  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  0  0  1  1  1  0  1  0  1  0  1  0  0  0  1  0  0  1  1  0  1  1  1  1  0  1  1  1  1  0  1  0  0  0  1  1  1  1  0  1  0  1  1  0  1  1  0  0  0  1  0  0  0  1  1  1  1  1  1  1  1  0  0  0  1  1  1  1  1  0  1  0  1  1  0  1  1  1  0  1  1  1  0  1  1  0  1  0  1  1  0  0  1  1  1  1  0  1  1  0  1  0  1  1  1  0  1  1  1  0  1  1  1  0  1  1  1  1  1  0  1  0  1  1  1  1  1  1  0  0  0  1  1  0  0  1  1  1  1  1  1  1  1  0  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  0  0  1  1  0  0  0  0  1  1  1  0  0  0  1  1  1  1  1  0  0  1  1  1  1  1  1  0  1  1  1  1  1  0  0  0  1  1  0  0  0  0  1  0  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  0  0  1  1  0  0  0  0  1  1  1  0  0  0  1  1  1  1  1  0  0  1  1  1  1  1  1  0  1  1  1  1  1  0  0  0  1  1  0  0  0  0  1  0  0  0  0  0  1  1  0  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  1  1  1  1  1  0  0  1  1  1  1  1  1  0  1  1  1  1  1  0  0  0  1  1  0  0  0  0  1  1  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  0  0  1  1  0  0  0  0  1  1  1  0  0  0  1  1  1  1  1  0  0  1  1  1  1  1  1  0  1  1  1  1  1  0  0  0  1  1  0  0  0  0  1  0  0  0  0  0  1  1  0  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  1  1  1  1  1  0  0  1  1  1  1  1  1  0  1  1  1  1  1  0  0  0  1  1  0  0  0  0  1  0  0  0  0  0  1  1  0  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  1  1  1  1  1  0  0  1  1  1  1  1  1  0  1  1  1  1  1  0  0  0  1  1  0  0  0  0  1  0  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  0  0  1  1  0  0  0  0  1  1  1  0  0  0  1  1  1  1  1  0  0  1  1  1  1  1  1  0  1  1  1  1  1  0  0  0  1  1  0  0  0  0  1  0  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  0  0  1  1  0  0  0  0  1  1  1  0  0  0  1  1  1  1  1  0  0  1  1  1  1  1  1  0  1  1  1  1  1  0  0  0  1  1  0  0  0  0  1  1  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  0  0  1  1  0  0  0  0  1  1  1  0  0  0  1  1  1  1  1  0  0  1  1  1  1  1  1  0  1  1  1  1  1  0  0  0  1  1  0  0  0  0  1  0  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  0  0  1  1  0  0  0  0  1  1  1  0  0  0  1  1  1  1  1  0  0  1  1  1  1  1  1  0  0  1  1  1  1  0  0  0  1  1  0  0  0  0'
raw_list = [int(i) for i in raw.split()]
gap_banknote_order['gap_cluster'] = raw_list
gap_banknote_order['filename'] = gap_banknote_order['UNIQID'].str.lstrip('r').astype('int')+1
gap_banknote_order = gap_banknote_order.drop(columns=['UNIQID'])

banknote_merged = pd.merge(banknote, gap_banknote_order[['gap_cluster', 'filename']], on='filename')
banknote_merged

,variance,skewness,curtosis,entropy,class,filename,gap_cluster
0,4.54590,8.16740,-2.4586,-1.46210,0,1,0
1,3.86600,-2.63830,1.9242,0.10645,0,2,1
2,3.45660,9.52280,-4.0112,-3.59440,0,3,0
3,0.32924,-4.45520,4.5718,-0.98880,0,4,0
4,4.36840,9.67180,-3.9606,-3.16250,0,5,1
...,...,...,...,...,...,...,...
1366,0.40614,1.34920,-1.4501,-0.55949,1,1367,1
1367,-1.38870,-4.87730,6.4774,0.34179,1,1368,1
1368,-3.75030,-13.45860,17.5932,-2.77710,1,1369,1
1369,-3.56370,-8.38270,12.3930,-1.28230,1,1370,1


In [35]:
print(banknote_merged['class'].value_counts())
print(banknote_merged['gap_cluster'].value_counts())


class
0    761
1    610
Name: count, dtype: int64
gap_cluster
1    855
0    516
Name: count, dtype: int64


In [36]:
pd.crosstab(banknote_merged['gap_cluster'], banknote_merged['class'])


class,0,1
gap_cluster,,
0,295,221
1,466,389


### clusters評估

In [37]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

ari = adjusted_rand_score(banknote_merged['class'], banknote_merged['gap_cluster'])
nmi = normalized_mutual_info_score(banknote_merged['class'], banknote_merged['gap_cluster'])

print("ARI:", ari)
print("NMI:", nmi)

ARI: -0.0014158644320544126
NMI: 0.0005017902126078439


### 重新命名GAP的clusters

In [38]:
import numpy as np
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import confusion_matrix

# true_y: 原始標籤 (字串或整數均可)
# gap_y : GAP 的 cluster label
conf = confusion_matrix(banknote_merged['class'], banknote_merged['gap_cluster'])
# Hungarian algorithm 找到最大化對角線和的對應
row_ind, col_ind = linear_sum_assignment(-conf)
mapping = {col: row for row, col in zip(row_ind, col_ind)}

# 重新命名 GAP 標籤，之後就跟真實標籤一致
banknote_merged['gap_cluster_aligned'] = np.vectorize(mapping.get)(banknote_merged['gap_cluster'])
acc = (banknote_merged['gap_cluster_aligned'] == banknote_merged['class']).mean()
print(f"GAP 分群與真實標籤對齊後的準確率：{acc:.4f}")


GAP 分群與真實標籤對齊後的準確率：0.5011


### 隨機森林模型評估 baseline

In [39]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 假設 'y' 是類別標籤，'gap_cluster' 是 GAP 分群結果
X = banknote_merged.drop(columns=['class', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
y = banknote_merged['class']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

banknote_baseline_model = RandomForestClassifier(random_state=42)
banknote_baseline_model.fit(X_train, y_train)
baseline_pred = banknote_baseline_model.predict(X_test)

print("=== Baseline 模型效能 ===")
print(classification_report(y_test, baseline_pred))


=== Baseline 模型效能 ===
              precision    recall  f1-score   support

           0       1.00      0.99      1.00       229
           1       0.99      1.00      0.99       183

    accuracy                           1.00       412
   macro avg       0.99      1.00      1.00       412
weighted avg       1.00      1.00      1.00       412



### 隨機森林模型評估 gap clusters

In [40]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 假設 'y' 是類別標籤，'gap_cluster' 是 GAP 分群結果
X = banknote_merged.drop(columns=['class', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
y = banknote_merged['gap_cluster']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.1, random_state=42)

banknote_gap_model = RandomForestClassifier(random_state=42)
banknote_gap_model.fit(X_train, y_train)
baseline_pred = banknote_gap_model.predict(X_test)

print("=== banknote_gap_model 模型效能 ===")
print(classification_report(y_test, baseline_pred))


=== banknote_gap_model 模型效能 ===
              precision    recall  f1-score   support

           0       0.45      0.33      0.38        52
           1       0.65      0.76      0.70        86

    accuracy                           0.59       138
   macro avg       0.55      0.54      0.54       138
weighted avg       0.57      0.59      0.58       138



### 隨機森林模型評估 gap 重新命名cluters

In [41]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 假設 'y' 是類別標籤，'gap_cluster' 是 GAP 分群結果
X = banknote_merged.drop(columns=['class', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
y = banknote_merged['gap_cluster_aligned']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.1, random_state=42)

banknote_gap_model = RandomForestClassifier(random_state=42)
banknote_gap_model.fit(X_train, y_train)
baseline_pred = banknote_gap_model.predict(X_test)

print("=== banknote_gap_aligned_model 模型效能 ===")
print(classification_report(y_test, baseline_pred))


=== banknote_gap_aligned_model 模型效能 ===
              precision    recall  f1-score   support

           0       0.64      0.79      0.71        86
           1       0.44      0.27      0.33        52

    accuracy                           0.59       138
   macro avg       0.54      0.53      0.52       138
weighted avg       0.56      0.59      0.57       138



### 隨機森林模型評估 弱集成強

In [42]:
banknote_grouped = banknote_merged.groupby('gap_cluster')

X = banknote_merged.drop(columns=['class', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
y = banknote_merged['gap_cluster_aligned']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.1, random_state=42)

from sklearn.ensemble import RandomForestClassifier
banknote_cluster_models={}
for c, sub_iris in banknote_grouped:
    X_sub = sub_iris.drop(columns=['class', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
    y_sub = sub_iris['class']
    rfc = RandomForestClassifier(random_state=42)
    model = rfc.fit(X_sub, y_sub)
    banknote_cluster_models[c] = model

banknote_router_model = RandomForestClassifier(random_state=42).fit(X_train, banknote_merged.loc[X_train.index, 'gap_cluster'])

def predict(x_new):
    cluster_id = banknote_router_model.predict(pd.DataFrame([x_new], columns=X_train.columns))[0]
    model = banknote_cluster_models[cluster_id]
    return model.predict(pd.DataFrame([x_new], columns=X_train.columns))[0]

banknote_ensemble_preds = [predict(row) for _, row in X_test.iterrows()]

from sklearn.metrics import classification_report

print("=== GAP 集成模型效能 ===")
print(classification_report(y_test, banknote_ensemble_preds))

=== GAP 集成模型效能 ===
              precision    recall  f1-score   support

           0       0.58      0.57      0.57        86
           1       0.30      0.31      0.30        52

    accuracy                           0.47       138
   macro avg       0.44      0.44      0.44       138
weighted avg       0.47      0.47      0.47       138



---
---

## Iris資料

In [43]:
import pandas as pd
iris = pd.read_csv(r"C:\Users\No\Documents\GitHub\psychic-spoon\DataSet\iris.csv", sep='\t')

iris['filename'] = [f'{i}' for i in range(1, 151)]
iris['filename'] = iris['filename'].astype(int)

gap_iris_order = pd.read_csv(r"C:\Users\No\Documents\GitHub\psychic-spoon\DataSet\gap_iris_order.txt", sep='\t')
values = [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
          2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
          0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1,
          0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1]
gap_iris_order['gap_cluster'] = values
gap_iris_order['filename'] = gap_iris_order['UNIQID'].str.lstrip('r').astype('int')+1
gap_iris_order = gap_iris_order.drop(columns=['UNIQID'])

iris_merged = pd.merge(iris, gap_iris_order[['gap_cluster', 'filename']], on='filename')

iris_merged['Species'] = iris_merged['Species'].replace({'setosa':0, 'versicolor':1, 'virginica':2})

C:\Users\No\AppData\Local\Temp\ipykernel_14764\2390759751.py:21: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  iris_merged['Species'] = iris_merged['Species'].replace({'setosa':0, 'versicolor':1, 'virginica':2})


### clusters 評估

In [44]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

ari = adjusted_rand_score(iris_merged['Species'], iris_merged['gap_cluster'])
nmi = normalized_mutual_info_score(iris_merged['Species'], iris_merged['gap_cluster'])

print("ARI:", ari)
print("NMI:", nmi)

ARI: 0.6727660175913543
NMI: 0.6404298764492126


In [45]:
import numpy as np
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import confusion_matrix

# true_y: 原始標籤 (字串或整數均可)
# gap_y : GAP 的 cluster label
conf = confusion_matrix(iris_merged['Species'], iris_merged['gap_cluster'])
# Hungarian algorithm 找到最大化對角線和的對應
row_ind, col_ind = linear_sum_assignment(-conf)
mapping = {col: row for row, col in zip(row_ind, col_ind)}

# 重新命名 GAP 標籤，之後就跟真實標籤一致
iris_merged['gap_cluster_aligned'] = np.vectorize(mapping.get)(iris_merged['gap_cluster'])


### 隨機森林模型評估 baseline

In [46]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 假設 'y' 是類別標籤，'gap_cluster' 是 GAP 分群結果
X = iris_merged.drop(columns=['Species', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
y = iris_merged['Species']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.1, random_state=42)

iris_baseline_model = RandomForestClassifier(random_state=42)
iris_baseline_model.fit(X_train, y_train)
iris_baseline_pred = iris_baseline_model.predict(X_test)

print("=== Baseline 模型效能 ===")
print(classification_report(y_test, iris_baseline_pred))


=== Baseline 模型效能 ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         5
           1       1.00      0.80      0.89         5
           2       0.83      1.00      0.91         5

    accuracy                           0.93        15
   macro avg       0.94      0.93      0.93        15
weighted avg       0.94      0.93      0.93        15



In [47]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 假設 'y' 是類別標籤，'gap_cluster' 是 GAP 分群結果
X = iris_merged.drop(columns=['Species', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
y = iris_merged['gap_cluster']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.1, random_state=42)

iris_gap_model = RandomForestClassifier(random_state=42)
iris_gap_model.fit(X_train, y_train)
iris_gap_pred = iris_gap_model.predict(X_test)

print("=== gap_model 模型效能 ===")
print(classification_report(y_test, iris_gap_pred))


=== gap_model 模型效能 ===
              precision    recall  f1-score   support

           0       1.00      0.80      0.89         5
           1       1.00      1.00      1.00         5
           2       0.83      1.00      0.91         5

    accuracy                           0.93        15
   macro avg       0.94      0.93      0.93        15
weighted avg       0.94      0.93      0.93        15



In [48]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 假設 'y' 是類別標籤，'gap_cluster' 是 GAP 分群結果
X = iris_merged.drop(columns=['Species', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
y = iris_merged['gap_cluster_aligned']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.1, random_state=42)

iris_gap_aligned_model = RandomForestClassifier(random_state=42)
iris_gap_aligned_model.fit(X_train, y_train)
iris_gap_aligned_pred = iris_gap_aligned_model.predict(X_test)

print("=== gap_aligned_model 模型效能 ===")
print(classification_report(y_test, iris_gap_aligned_pred))


=== gap_aligned_model 模型效能 ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         5
           1       0.83      1.00      0.91         5
           2       1.00      0.80      0.89         5

    accuracy                           0.93        15
   macro avg       0.94      0.93      0.93        15
weighted avg       0.94      0.93      0.93        15



### 隨機森林模型評估 弱集成強

In [49]:
iris_grouped = iris_merged.groupby('gap_cluster')

from sklearn.ensemble import RandomForestClassifier
iris_cluster_models={}
for c, sub_iris in iris_grouped:
    X_sub = sub_iris.drop(columns=['Species', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
    y_sub = sub_iris['Species']
    rfc = RandomForestClassifier(random_state=42)
    model = rfc.fit(X_sub, y_sub)
    iris_cluster_models[c] = model

iris_router_model = RandomForestClassifier(random_state=42).fit(X_train, iris_merged.loc[X_train.index, 'gap_cluster'])

def predict(x_new):
    cluster_id = iris_router_model.predict(pd.DataFrame([x_new], columns=X_train.columns))[0]
    model = iris_cluster_models[cluster_id]
    return model.predict(pd.DataFrame([x_new], columns=X_train.columns))[0]

iris_ensemble_preds = [predict(row) for _, row in X_test.iterrows()]

print("=== GAP 集成模型效能 ===")
print(classification_report(y_test, iris_ensemble_preds))

=== GAP 集成模型效能 ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         5
           1       0.83      1.00      0.91         5
           2       1.00      0.80      0.89         5

    accuracy                           0.93        15
   macro avg       0.94      0.93      0.93        15
weighted avg       0.94      0.93      0.93        15



---
---

## Secom 資料

In [50]:
import pandas as pd
secom = pd.read_csv(r"C:\Users\No\Documents\GitHub\psychic-spoon\DataSet\secom.csv", sep='\t').fillna(0)
secom['filename'] = [f'{i}' for i in range(1, 1568)]
secom['filename'] = secom['filename'].astype(int)
secom['Pass/Fail'] = secom['Pass/Fail'].replace({-1:0})

In [51]:
gap_secom_order = pd.read_csv(r"C:\Users\No\Documents\GitHub\psychic-spoon\DataSet\gap_secom_order.txt", sep='\t')
raw='1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  0  1  1  1  0  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  0  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  0  1  1  1  1  0  1  1  1  1  1  1  1  0  1  1  1  0  0  1  1  1  0  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  0  1  1  1  1  1  1  1  0  1  0  1  1  1  1  1  0  1  1  0  1  1  1  1  1  1  0  0  1  0  1  0  0  0  1  1  1  1  0  1  1  1  1  1  0  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  0  1  1  1  0  1  0  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  0  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  0  1  1  0  0  1  1  1  1  1  0  0  1  0  0  1  1  1  1  1  1  1  1  0  0  1  1  1  1  1  0  0  1  0  0  1  1  1  1  0  1  0  1  1  1  1  1  0  1  1  1  1  1  1  1  1  0  0  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  1  0  1  1  0  1  1  1  0  1  1  0  1  1  1  0  1  1  1  1  1  1  1  0  1  1  1  1  1  0  0  0  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  0  1  1  1  1  0  1  1  1  1  0  0  1  0  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  0  1  1  1  1  0  0  1  1  1  1  1  1  1  1  0  0  0  1  1  1  1  1  1  1  0  1  1  1  1  0  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  0  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  0  1  1  1  1  0  1  1  1  1  1  1  1  0  1  1  1  1  1  0  1  1  0  1  0  0  1  1  0  1  0  1  1  0  1  1  0  1  1  1  1  0  1  1  1  1  1  0  1  1  1  1  1  1  0  1  1  0  1  0  1  1  1  1  1  1  1  0  0  0  1  1  1  1  0  0  1  1  1  1  0  0  0  1  1  1  0  1  0  1  1  1  1  1  0  1  0  1  0  0  1  1  1  0  1  0  0  1  1  0  1  0  1  1  1  1  1  0  0  1  1  1  1  1  1  0  0  1  0  1  1  1  0  1  1  1  0  0  0  0  0  0  1  0  1  1  1  1  1  1  1  0  1  1  1  0  0  0  0  1  1  1  1  1  0  1  1  0  1  1  0  1  1  1  1  1  1  1  0  1  0  1  1  1  0  1  1  0  1  1  1  1  0  0  1  0  1  1  0  0  1  1  1  0  0  1  1  1  1  1  0  0  0  0  1  1  1  0  1  0  0  1  0  1  1  0  1  1  0  0  0  0  0  1  1  1  0  0  1  1  0  0  1  1  1  1  1  0  1  0  1  1  0  1  0  1  0  1  1  1  1  1  0  1  0  1  1  1  1  1  1  1  1  0  1  1  1  1  0  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  0  1  0  1  1  0  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  0  0  1  0  1  1  1  1  1  1  0  0  1  1  1  1  1  1  1  1  0  1  0  1  1  1  1  0  1  1  1  1  0  1  0  1  1  0  1  1  1  0  1  0  0  0  1  1  0  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  0  1  0  1  1  0  1  1  1  1  0  1  1  0  0  0  1  1  1  1  1  1  0  1  1  0  0  0  1  1  1  0  1  1  0  1  0  0  1  1  1  1  1  1  1  1  1  0  0  0  1  1  1  1  1  0  0  1  1  1  0  1  1  1  1  1  0  1  0  1  1  1  0  0  0  1  0  0  1  1  0  1  1  1  1  0  0  0  1  0  1  1  0  0  1  0  0  1  1  1  1  1  0  1  1  1  1  1  1  1  1  0  1  1  0  1  0  0  1  1  1  1  1  1  1  0  1  0  0  1  1  0  0  0  1  1  1  1  0  1  1  1  0  1  0  1  1  1  0  1  1  0  0  1  0  1  0  1  0  1  0  0  0  1  1  1  1  1  0  0  0  0  0  0  1  1  1  1  1  1  1  0  0  1  1  0  0  1  1  1  1  1  0  1  1  0  1  1  1  1  0  1  1  1  0  1  1  0  0  1  1  1  0  1  0  1  1  1  1  1  1  1  1  1  0  1  1  1  1  0  1  0  0  1  1  0  1  0  1  1  1  0  0  1  1  1  0  1  1  0  0  1  1  0  0  1  1  0  1  0  1  0  0  0  1  1  1  1  1  1  0  1  1  1  1  1  0  1  0  1  1  1  1  0  1  0  1  1  1  1  0  1  0  0  1  1  0  0  1  1  1  0  1  1  1  0  1  0  1  1  1  1  1  1  1  0  1  1  1  0  1  0  1  0  1  0  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  0  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  0  1  1  0  0  0  1  1  1  1  0  1  1  0  0  0  0  1  1  1  1  0  0  1  1  1  1  1  0  1  0  0  1  1  1  1  1  1  0  1  1  0  0  1  1  1  1  1  0  1  1  1  1  1  0  1  0  1  1  1  1  1  1  1  1  1  1  1  0  0  1  1  1  1  0  1  1  1  1  0  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1'
raw_list = [int(i) for i in raw.split()]
gap_secom_order['gap_cluster'] = raw_list
gap_secom_order['filename'] = gap_secom_order['UNIQID'].str.lstrip('r').astype('int')+1
gap_secom_order = gap_secom_order.drop(columns=['UNIQID'])

secom_merged = pd.merge(secom, gap_secom_order[['gap_cluster', 'filename']], on='filename')
secom_merged

,Time,x1,x2,x3,x4,x5,x6,x7,x8,x9,...,x584,x585,x586,x587,x588,x589,x590,Pass/Fail,filename,gap_cluster
0,2008-07-19 11:55:00,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,...,0.0118,0.0035,2.3630,0.0000,0.0000,0.0000,0.0000,0,1,1
1,2008-07-19 12:32:00,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,...,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,0,2,1
2,2008-07-19 13:17:00,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,...,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1,3,0
3,2008-07-19 14:43:00,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,...,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,0,4,0
4,2008-07-19 15:22:00,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,...,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,0,5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1562,2008-10-16 15:13:00,2899.41,2464.36,2179.7333,3085.3781,1.4843,100.0,82.2467,0.1248,1.3424,...,0.0143,0.0039,2.8669,0.0068,0.0138,0.0047,203.1720,0,1563,1
1563,2008-10-16 20:49:00,3052.31,2522.55,2198.5667,1124.6595,0.8763,100.0,98.4689,0.1205,1.4333,...,0.0131,0.0036,2.6238,0.0068,0.0138,0.0047,203.1720,0,1564,1
1564,2008-10-17 05:26:00,2978.81,2379.78,2206.3000,1110.4967,0.8236,100.0,99.4122,0.1208,0.0000,...,0.0153,0.0041,3.0590,0.0197,0.0086,0.0025,43.5231,0,1565,1
1565,2008-10-17 06:01:00,2894.92,2532.01,2177.0333,1183.7287,1.5726,100.0,98.7978,0.1213,1.4622,...,0.0178,0.0038,3.5662,0.0262,0.0245,0.0075,93.4941,0,1566,1


In [52]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

ari = adjusted_rand_score(secom_merged['Pass/Fail'], secom_merged['gap_cluster'])
nmi = normalized_mutual_info_score(secom_merged['Pass/Fail'], secom_merged['gap_cluster'])

print("ARI:", ari)
print("NMI:", nmi)

ARI: -0.002875766693333256
NMI: 3.8116800882744636e-05


可能原因

Pass/Fail 與特徵空間本就無明顯分群
→ 資料可能需要額外特徵工程或不同維度來描述良率。

群數 (k) 設定不合適
→ 可依 GAP 的 AR score 或用穩定度評估再選 k。

資料尺度 / 雜訊影響
→ 先標準化、去除高相關或高噪聲變數再試。

類別高度不平衡
→ ARI 對極端不平衡較敏感；可先檢查 Pass/Fail 分布。

In [53]:
pd.crosstab(secom_merged['gap_cluster'], secom_merged['Pass/Fail'])


Pass/Fail,0,1
gap_cluster,,
0,337,23
1,1126,81


In [54]:
import numpy as np
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import confusion_matrix

# true_y: 原始標籤 (字串或整數均可)
# gap_y : GAP 的 cluster label
conf = confusion_matrix(secom_merged['Pass/Fail'], secom_merged['gap_cluster'])
# Hungarian algorithm 找到最大化對角線和的對應
row_ind, col_ind = linear_sum_assignment(-conf)
mapping = {col: row for row, col in zip(row_ind, col_ind)}

# 重新命名 GAP 標籤，之後就跟真實標籤一致
secom_merged['gap_cluster_aligned'] = np.vectorize(mapping.get)(secom_merged['gap_cluster'])
secom_merged

,Time,x1,x2,x3,x4,x5,x6,x7,x8,x9,...,x585,x586,x587,x588,x589,x590,Pass/Fail,filename,gap_cluster,gap_cluster_aligned
0,2008-07-19 11:55:00,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,...,0.0035,2.3630,0.0000,0.0000,0.0000,0.0000,0,1,1,0
1,2008-07-19 12:32:00,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,...,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,0,2,1,0
2,2008-07-19 13:17:00,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,...,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1,3,0,1
3,2008-07-19 14:43:00,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,...,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,0,4,0,1
4,2008-07-19 15:22:00,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,...,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,0,5,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1562,2008-10-16 15:13:00,2899.41,2464.36,2179.7333,3085.3781,1.4843,100.0,82.2467,0.1248,1.3424,...,0.0039,2.8669,0.0068,0.0138,0.0047,203.1720,0,1563,1,0
1563,2008-10-16 20:49:00,3052.31,2522.55,2198.5667,1124.6595,0.8763,100.0,98.4689,0.1205,1.4333,...,0.0036,2.6238,0.0068,0.0138,0.0047,203.1720,0,1564,1,0
1564,2008-10-17 05:26:00,2978.81,2379.78,2206.3000,1110.4967,0.8236,100.0,99.4122,0.1208,0.0000,...,0.0041,3.0590,0.0197,0.0086,0.0025,43.5231,0,1565,1,0
1565,2008-10-17 06:01:00,2894.92,2532.01,2177.0333,1183.7287,1.5726,100.0,98.7978,0.1213,1.4622,...,0.0038,3.5662,0.0262,0.0245,0.0075,93.4941,0,1566,1,0


In [55]:
secom_merged['Pass/Fail'].value_counts()

Pass/Fail
0    1463
1     104
Name: count, dtype: int64

In [56]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 假設 'y' 是類別標籤，'gap_cluster' 是 GAP 分群結果
X = secom_merged.drop(columns=['Time', 'Pass/Fail', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
y = secom_merged['Pass/Fail']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

secom_baseline_model = RandomForestClassifier(random_state=42)
secom_baseline_model.fit(X_train, y_train)
secom_baseline_pred = secom_baseline_model.predict(X_test)

print("=== Baseline 模型效能 ===")
print(classification_report(y_test, secom_baseline_pred))


=== Baseline 模型效能 ===
              precision    recall  f1-score   support

           0       0.93      1.00      0.96       440
           1       0.00      0.00      0.00        31

    accuracy                           0.93       471
   macro avg       0.47      0.50      0.48       471
weighted avg       0.87      0.93      0.90       471



In [ ]:
ratio   = y_train.value_counts()[0] / y_train.value_counts()[1]

In [ ]:
from xgboost import XGBClassifier
X = secom_merged.drop(columns=['Time', 'Pass/Fail', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
y = secom_merged['Pass/Fail']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)
xgbcl = XGBClassifier(random_state=42)
secom_xgb_model = xgbcl.fit(X_train, y_train)
secom_xgb_pred = secom_xgb_model.predict(X_test)

print(classification_report(y_test, secom_xgb_pred))

              precision    recall  f1-score   support

           0       0.94      1.00      0.97       440
           1       0.50      0.06      0.11        31

    accuracy                           0.93       471
   macro avg       0.72      0.53      0.54       471
weighted avg       0.91      0.93      0.91       471



In [88]:
from xgboost import XGBClassifier
X = secom_merged.drop(columns=['Time', 'Pass/Fail', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
y = secom_merged['Pass/Fail']
ratio   = y_train.value_counts()[0] / y_train.value_counts()[1] 
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)
xgbcl = XGBClassifier(random_state=42, scale_pos_weight=ratio, eval_metric='aucpr')
secom_xgb_model = xgbcl.fit(X_train, y_train)
secom_xgb_pred = secom_xgb_model.predict(X_test)

print(classification_report(y_test, secom_xgb_pred))

              precision    recall  f1-score   support

           0       0.94      0.99      0.96       440
           1       0.25      0.03      0.06        31

    accuracy                           0.93       471
   macro avg       0.59      0.51      0.51       471
weighted avg       0.89      0.93      0.90       471



In [57]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 假設 'y' 是類別標籤，'gap_cluster' 是 GAP 分群結果
X = secom_merged.drop(columns=['Time', 'Pass/Fail', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
y = secom_merged['gap_cluster']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

secom_gap_model = RandomForestClassifier(random_state=42)
secom_gap_model.fit(X_train, y_train)
secom_gap_pred = secom_gap_model.predict(X_test)

print("=== gap_model 模型效能 ===")
print(classification_report(y_test, secom_gap_pred))


=== gap_model 模型效能 ===
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       108
           1       0.77      1.00      0.87       363

    accuracy                           0.77       471
   macro avg       0.39      0.50      0.44       471
weighted avg       0.59      0.77      0.67       471



c:\Users\No\anaconda3\envs\Laptop_20250527\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\No\anaconda3\envs\Laptop_20250527\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\No\anaconda3\envs\Laptop_20250527\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is",

In [58]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 假設 'y' 是類別標籤，'gap_cluster' 是 GAP 分群結果
X = secom_merged.drop(columns=['Time', 'Pass/Fail', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
y = secom_merged['gap_cluster_aligned']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.1, random_state=42)

secom_gap_aligned_model = RandomForestClassifier(random_state=42)
secom_gap_aligned_model.fit(X_train, y_train)
secom_gap_aligned_pred = secom_gap_aligned_model.predict(X_test)

print("=== gap_aligned_model 模型效能 ===")
print(classification_report(y_test, secom_gap_aligned_pred))


=== gap_aligned_model 模型效能 ===
              precision    recall  f1-score   support

           0       0.77      1.00      0.87       121
           1       0.00      0.00      0.00        36

    accuracy                           0.77       157
   macro avg       0.39      0.50      0.44       157
weighted avg       0.59      0.77      0.67       157



c:\Users\No\anaconda3\envs\Laptop_20250527\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\No\anaconda3\envs\Laptop_20250527\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\No\anaconda3\envs\Laptop_20250527\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is",

In [79]:
from xgboost import XGBClassifier
X = secom_merged.drop(columns=['Time', 'Pass/Fail', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
y = secom_merged['gap_cluster_aligned']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)
secom_xgb_gap_aligned_model = XGBClassifier(random_state=42)
secom_xgb_gap_aligned_model.fit(X_train, y_train)
secom_xgb_gap_aligned_pred = secom_xgb_gap_aligned_model.predict(X_test)

print(classification_report(y_test, secom_xgb_gap_aligned_pred))

              precision    recall  f1-score   support

           0       0.77      0.96      0.85       363
           1       0.20      0.04      0.06       108

    accuracy                           0.75       471
   macro avg       0.48      0.50      0.46       471
weighted avg       0.64      0.75      0.67       471



In [59]:
secom_grouped = secom_merged.groupby('gap_cluster')

from sklearn.ensemble import RandomForestClassifier
secom_cluster_models={}
for c, sub_secom in secom_grouped:
    X_sub = sub_secom.drop(columns=['Time', 'Pass/Fail', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
    y_sub = sub_secom['Pass/Fail']
    rfc = RandomForestClassifier(random_state=42)
    model = rfc.fit(X_sub, y_sub)
    secom_cluster_models[c] = model

secom_router_model = RandomForestClassifier(random_state=42).fit(X_train, secom_merged.loc[X_train.index, 'gap_cluster'])

def predict(x_new):
    cluster_id = secom_router_model.predict(pd.DataFrame([x_new], columns=X_train.columns))[0]
    model = secom_cluster_models[cluster_id]
    return model.predict(pd.DataFrame([x_new], columns=X_train.columns))[0]

secom_ensemble_preds = [predict(row) for _, row in X_test.iterrows()]

print("=== GAP 集成模型效能 ===")
print(classification_report(y_test, secom_ensemble_preds))

=== GAP 集成模型效能 ===
              precision    recall  f1-score   support

           0       0.76      0.94      0.84       121
           1       0.00      0.00      0.00        36

    accuracy                           0.73       157
   macro avg       0.38      0.47      0.42       157
weighted avg       0.59      0.73      0.65       157



---
---

In [60]:
import pandas as pd

customer = pd.read_csv(r"C:\Users\No\Documents\GitHub\psychic-spoon\DataSet\customer.csv", sep='\t')
# customer = customer.drop(columns=['Churn']) 
# customer.to_csv('customer_no_y.csv', sep='\t', encoding='utf-8', index=False) 

In [61]:
customer

,Call Failure,Complains,Subscription Length,Charge Amount,Seconds of Use,Frequency of use,Frequency of SMS,Distinct Called Numbers,Age Group,Tariff Plan,Status,Age,Customer Value,Churn
0,8,0,38,0,4370,71,5,17,3,1,1,30,197.640,0
1,0,0,39,0,318,5,7,4,2,1,2,25,46.035,0
2,10,0,37,0,2453,60,359,24,3,1,1,30,1536.520,0
3,10,0,38,0,4198,66,1,35,1,1,1,15,240.020,0
4,3,0,38,0,2393,58,2,33,1,1,1,15,145.805,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3145,21,0,19,2,6697,147,92,44,2,2,1,25,721.980,0
3146,17,0,17,1,9237,177,80,42,5,1,1,55,261.210,0
3147,13,0,18,4,3157,51,38,21,3,1,1,30,280.320,0
3148,7,0,11,2,4695,46,222,12,3,1,1,30,1077.640,0


In [62]:
customer['filename'] = [f'{i}' for i in range(1, 3151)]
customer['filename'] = customer['filename'].astype(int)
gap_customer_order = pd.read_csv(r"C:\Users\No\Documents\GitHub\psychic-spoon\DataSet\gap_customer_order.txt", sep='\t')
raw ='1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  0  0  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  0  1  0  0  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  0  0  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  0  0  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  0  1  1  1  1  1  1  1  1  1  0  1  1  0  0  1  1  1  1  1  1  1  0  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  1  0  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  0  0  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  0  0  1  1  1  1  1  1  1  1  1  0  0  1  0  0  1  1  1  1  1  1  1  0  0  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  0  0  1  1  1  1  1  1  1  1  1  0  0  1  0  0  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  0  0  1  1  1  1  1  1  1  1  1  0  0  1  0  0  1  1  1  1  1  1  1  0  0  0  1  0  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  0  0  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  0  0  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  0  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  0  0  1  1  1  0  1  1  1  1  1  0  1  1  0  0  1  1  1  1  1  0  1  0  0  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  0  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1'
raw_list = [int(i) for i in raw.split()]
gap_customer_order['gap_cluster'] = raw_list
gap_customer_order['filename'] = gap_customer_order['UNIQID'].str.lstrip('r').astype('int')+1
gap_customer_order = gap_customer_order.drop(columns=['UNIQID'])

customer_merged = pd.merge(customer, gap_customer_order[['gap_cluster', 'filename']], on='filename')
customer_merged

,Call Failure,Complains,Subscription Length,Charge Amount,Seconds of Use,Frequency of use,Frequency of SMS,Distinct Called Numbers,Age Group,Tariff Plan,Status,Age,Customer Value,Churn,filename,gap_cluster
0,8,0,38,0,4370,71,5,17,3,1,1,30,197.640,0,1,1
1,0,0,39,0,318,5,7,4,2,1,2,25,46.035,0,2,1
2,10,0,37,0,2453,60,359,24,3,1,1,30,1536.520,0,3,1
3,10,0,38,0,4198,66,1,35,1,1,1,15,240.020,0,4,1
4,3,0,38,0,2393,58,2,33,1,1,1,15,145.805,0,5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3145,21,0,19,2,6697,147,92,44,2,2,1,25,721.980,0,3146,1
3146,17,0,17,1,9237,177,80,42,5,1,1,55,261.210,0,3147,0
3147,13,0,18,4,3157,51,38,21,3,1,1,30,280.320,0,3148,1
3148,7,0,11,2,4695,46,222,12,3,1,1,30,1077.640,0,3149,1


In [63]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

ari = adjusted_rand_score(customer_merged['Churn'], customer_merged['gap_cluster'])
nmi = normalized_mutual_info_score(customer_merged['Churn'], customer_merged['gap_cluster'])

print("ARI:", ari)
print("NMI:", nmi)

ARI: -0.020055494103854262
NMI: 0.00175291571684074


linear_sum_assignment
$$
assignment　cost = min\sum_i\sum_jC_{i,j}X_{i,j}
$$

In [64]:
import numpy as np
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import confusion_matrix

# true_y: 原始標籤 (字串或整數均可)
# gap_y : GAP 的 cluster label
conf = confusion_matrix(customer_merged['Churn'], customer_merged['gap_cluster'])
# Hungarian algorithm 找到最大化對角線和的對應
row_ind, col_ind = linear_sum_assignment(-conf)
mapping = {col: row for row, col in zip(row_ind, col_ind)}

# 重新命名 GAP 標籤，之後就跟真實標籤一致
customer_merged['gap_cluster_aligned'] = np.vectorize(mapping.get)(customer_merged['gap_cluster'])
customer_merged

,Call Failure,Complains,Subscription Length,Charge Amount,Seconds of Use,Frequency of use,Frequency of SMS,Distinct Called Numbers,Age Group,Tariff Plan,Status,Age,Customer Value,Churn,filename,gap_cluster,gap_cluster_aligned
0,8,0,38,0,4370,71,5,17,3,1,1,30,197.640,0,1,1,0
1,0,0,39,0,318,5,7,4,2,1,2,25,46.035,0,2,1,0
2,10,0,37,0,2453,60,359,24,3,1,1,30,1536.520,0,3,1,0
3,10,0,38,0,4198,66,1,35,1,1,1,15,240.020,0,4,1,0
4,3,0,38,0,2393,58,2,33,1,1,1,15,145.805,0,5,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3145,21,0,19,2,6697,147,92,44,2,2,1,25,721.980,0,3146,1,0
3146,17,0,17,1,9237,177,80,42,5,1,1,55,261.210,0,3147,0,1
3147,13,0,18,4,3157,51,38,21,3,1,1,30,280.320,0,3148,1,0
3148,7,0,11,2,4695,46,222,12,3,1,1,30,1077.640,0,3149,1,0


In [65]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 假設 'y' 是類別標籤，'gap_cluster' 是 GAP 分群結果
X = customer_merged.drop(columns=['Churn', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
y = customer_merged['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

customer_baseline_model = RandomForestClassifier(random_state=42)
customer_baseline_model.fit(X_train, y_train)
customer_baseline_pred = customer_baseline_model.predict(X_test)

print("=== Baseline 模型效能 ===")
print(classification_report(y_test, customer_baseline_pred))


=== Baseline 模型效能 ===
              precision    recall  f1-score   support

           0       0.96      0.98      0.97       797
           1       0.89      0.80      0.84       148

    accuracy                           0.95       945
   macro avg       0.93      0.89      0.91       945
weighted avg       0.95      0.95      0.95       945



In [66]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 假設 'y' 是類別標籤，'gap_cluster' 是 GAP 分群結果
X = customer_merged.drop(columns=['Churn', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
y = customer_merged['gap_cluster']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

customer_baseline_model = RandomForestClassifier(random_state=42)
customer_baseline_model.fit(X_train, y_train)
customer_baseline_pred = customer_baseline_model.predict(X_test)

print("=== gap_model 模型效能 ===")
print(classification_report(y_test, customer_baseline_pred))


=== gap_model 模型效能 ===
              precision    recall  f1-score   support

           0       0.14      0.02      0.04        48
           1       0.95      0.99      0.97       897

    accuracy                           0.94       945
   macro avg       0.55      0.51      0.50       945
weighted avg       0.91      0.94      0.92       945



In [67]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 假設 'y' 是類別標籤，'gap_cluster' 是 GAP 分群結果
X = customer_merged.drop(columns=['Churn', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
y = customer_merged['gap_cluster_aligned']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

customer_gap_aligned_model = RandomForestClassifier(random_state=42)
customer_gap_aligned_model.fit(X_train, y_train)
customer_gap_aligned_pred = customer_gap_aligned_model.predict(X_test)

print("=== gap_aligned_model 模型效能 ===")
print(classification_report(y_test, customer_gap_aligned_pred))


=== gap_aligned_model 模型效能 ===
              precision    recall  f1-score   support

           0       0.95      1.00      0.97       897
           1       1.00      0.02      0.04        48

    accuracy                           0.95       945
   macro avg       0.98      0.51      0.51       945
weighted avg       0.95      0.95      0.93       945



In [72]:
print(customer_merged['Churn'].value_counts())
print(customer_merged['gap_cluster'].value_counts())

Churn
0    2655
1     495
Name: count, dtype: int64
gap_cluster
1    2991
0     159
Name: count, dtype: int64


In [68]:
customer_grouped = customer_merged.groupby('gap_cluster')

from sklearn.ensemble import RandomForestClassifier
customer_cluster_models={}
for c, sub_customer in customer_grouped:
    X_sub = sub_customer.drop(columns=['Churn', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
    y_sub = sub_customer['Churn']
    rfc = RandomForestClassifier(random_state=42)
    model = rfc.fit(X_sub, y_sub)
    customer_cluster_models[c] = model

customer_router_model = RandomForestClassifier(random_state=42).fit(X_train, customer_merged.loc[X_train.index, 'gap_cluster'])

def predict(x_new):
    cluster_id = customer_router_model.predict(pd.DataFrame([x_new], columns=X_train.columns))[0]
    model = customer_cluster_models[cluster_id]
    return model.predict(pd.DataFrame([x_new], columns=X_train.columns))[0]

customer_ensemble_preds = [predict(row) for _, row in X_test.iterrows()]

print("=== GAP 集成模型效能 ===")
print(classification_report(y_test, customer_ensemble_preds))

=== GAP 集成模型效能 ===
              precision    recall  f1-score   support

           0       0.94      0.84      0.89       897
           1       0.03      0.08      0.04        48

    accuracy                           0.80       945
   macro avg       0.49      0.46      0.46       945
weighted avg       0.90      0.80      0.84       945

